In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# تحميل البيانات (كما في الكود الأصلي)
X_train = pd.read_csv('X_train_transformed.csv')
X_test = pd.read_csv('X_test_transformed.csv')
y_train_clf = pd.read_csv('y_train_clf.csv').squeeze()
y_test_clf = pd.read_csv('y_test_clf.csv').squeeze()
y_train_reg = pd.read_csv('y_train_reg.csv').squeeze()
y_test_reg = pd.read_csv('y_test_reg.csv').squeeze()

# إزالة عمود Overall_Rating للتصنيف (لتجنب تسرب البيانات)
X_train_clf = X_train.drop('num__Overall_Rating', axis=1)
X_test_clf = X_test.drop('num__Overall_Rating', axis=1)
# للانحدار نتركه كما هو (نريد توقع القيمة، وليست فئات)

In [ ]:
from sklearn.svm import SVC

# تعريف مجموعة المعلمات
param_grid_svc = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1],
    'kernel': ['rbf', 'linear', 'poly']
}

# GridSearchCV مع 5-fold StratifiedKFold
svc_grid = GridSearchCV(
    SVC(random_state=42),
    param_grid_svc,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

svc_grid.fit(X_train_clf, y_train_clf)

print("Best SVC Parameters:", svc_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(svc_grid.best_score_))

# تقييم أفضل نموذج على مجموعة الاختبار
best_svc = svc_grid.best_estimator_
y_pred_svc = best_svc.predict(X_test_clf)
print("\nSVC Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_svc)))
print(classification_report(y_test_clf, y_pred_svc))

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
from sklearn.svm import SVR

param_grid_svr = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1],
    'kernel': ['rbf', 'linear', 'poly']
}

svr_grid = GridSearchCV(
    SVR(),
    param_grid_svr,
    cv=KFold(5),
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

svr_grid.fit(X_train, y_train_reg)

print("Best SVR Parameters:", svr_grid.best_params_)
print("Best CV R²: {:.4f}".format(svr_grid.best_score_))

best_svr = svr_grid.best_estimator_
y_pred_svr = best_svr.predict(X_test)
print("\nSVR Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_svr)))
print("SVR Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_svr))))

In [10]:
from sklearn.neighbors import KNeighborsClassifier

param_grid_knn_clf = {
    'n_neighbors': list(range(3, 102, 2)),  # أعداد فردية
    'metric': ['euclidean', 'manhattan']
}

knn_clf_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn_clf,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

knn_clf_grid.fit(X_train_clf, y_train_clf)

print("Best KNN Classifier Parameters:", knn_clf_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(knn_clf_grid.best_score_))

best_knn_clf = knn_clf_grid.best_estimator_
y_pred_knn_clf = best_knn_clf.predict(X_test_clf)
print("\nKNN Classifier Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_knn_clf)))
print(classification_report(y_test_clf, y_pred_knn_clf))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best KNN Classifier Parameters: {'metric': 'manhattan', 'n_neighbors': 5}
Best CV Accuracy: 0.8585

KNN Classifier Test Accuracy: 0.8610
              precision    recall  f1-score   support

       Elite       0.86      0.59      0.70       105
        High       0.85      0.81      0.83       716
         Low       0.89      0.87      0.88      1283
         Mid       0.85      0.89      0.87      1830

    accuracy                           0.86      3934
   macro avg       0.86      0.79      0.82      3934
weighted avg       0.86      0.86      0.86      3934



In [11]:
from sklearn.neighbors import KNeighborsRegressor

param_grid_knn_reg = {
    'n_neighbors': list(range(3, 102, 2)),
    'metric': ['euclidean', 'manhattan']
}

knn_reg_grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid_knn_reg,
    cv=KFold(5),
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

knn_reg_grid.fit(X_train, y_train_reg)

print("Best KNN Regressor Parameters:", knn_reg_grid.best_params_)
print("Best CV R²: {:.4f}".format(knn_reg_grid.best_score_))

best_knn_reg = knn_reg_grid.best_estimator_
y_pred_knn_reg = best_knn_reg.predict(X_test)
print("\nKNN Regressor Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_knn_reg)))
print("KNN Regressor Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_knn_reg))))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best KNN Regressor Parameters: {'metric': 'euclidean', 'n_neighbors': 3}
Best CV R²: 0.8809

KNN Regressor Test R²: 0.8844
KNN Regressor Test RMSE: 2.1558


In [12]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# تحويل تسميات التصنيف إلى أرقام (XGBoost يتطلب أرقاماً)
le = LabelEncoder()
y_train_clf_enc = le.fit_transform(y_train_clf)
y_test_clf_enc = le.transform(y_test_clf)

param_grid_xgb_clf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

xgb_clf_grid = GridSearchCV(
    XGBClassifier(eval_metric='mlogloss', random_state=42),
    param_grid_xgb_clf,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

xgb_clf_grid.fit(X_train_clf, y_train_clf_enc)

print("Best XGBoost Classifier Parameters:", xgb_clf_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(xgb_clf_grid.best_score_))

best_xgb_clf = xgb_clf_grid.best_estimator_
y_pred_xgb_clf_enc = best_xgb_clf.predict(X_test_clf)
y_pred_xgb_clf = le.inverse_transform(y_pred_xgb_clf_enc)
print("\nXGBoost Classifier Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_xgb_clf)))
print(classification_report(y_test_clf, y_pred_xgb_clf))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best XGBoost Classifier Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Best CV Accuracy: 0.9149

XGBoost Classifier Test Accuracy: 0.9123
              precision    recall  f1-score   support

       Elite       0.93      0.88      0.90       105
        High       0.91      0.93      0.92       716
         Low       0.90      0.92      0.91      1283
         Mid       0.92      0.90      0.91      1830

    accuracy                           0.91      3934
   macro avg       0.92      0.91      0.91      3934
weighted avg       0.91      0.91      0.91      3934



In [13]:
from xgboost import XGBRegressor

param_grid_xgb_reg = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

xgb_reg_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid_xgb_reg,
    cv=KFold(5),
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

xgb_reg_grid.fit(X_train, y_train_reg)

print("Best XGBoost Regressor Parameters:", xgb_reg_grid.best_params_)
print("Best CV R²: {:.4f}".format(xgb_reg_grid.best_score_))

best_xgb_reg = xgb_reg_grid.best_estimator_
y_pred_xgb_reg = best_xgb_reg.predict(X_test)
print("\nXGBoost Regressor Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_xgb_reg)))
print("XGBoost Regressor Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_xgb_reg))))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best XGBoost Regressor Parameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
Best CV R²: 0.9756

XGBoost Regressor Test R²: 0.9770
XGBoost Regressor Test RMSE: 0.9616


In [14]:
print("\n" + "="*60)
print("SUMMARY OF GRID SEARCH RESULTS")
print("="*60)
print(f"SVC           | Best Params: {svc_grid.best_params_} | Test Acc: {accuracy_score(y_test_clf, y_pred_svc):.4f}")
print(f"SVR           | Best Params: {svr_grid.best_params_} | Test R²: {r2_score(y_test_reg, y_pred_svr):.4f}")
print(f"KNN Classifier| Best Params: {knn_clf_grid.best_params_} | Test Acc: {accuracy_score(y_test_clf, y_pred_knn_clf):.4f}")
print(f"KNN Regressor | Best Params: {knn_reg_grid.best_params_} | Test R²: {r2_score(y_test_reg, y_pred_knn_reg):.4f}")
print(f"XGBoost Classifier| Best Params: {xgb_clf_grid.best_params_} | Test Acc: {accuracy_score(y_test_clf, y_pred_xgb_clf):.4f}")
print(f"XGBoost Regressor| Best Params: {xgb_reg_grid.best_params_} | Test R²: {r2_score(y_test_reg, y_pred_xgb_reg):.4f}")


SUMMARY OF GRID SEARCH RESULTS


NameError: name 'svc_grid' is not defined

In [ ]:
....

SyntaxError: invalid syntax (1805539695.py, line 1)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# تحميل البيانات (كما في كودك الأصلي)
X_train = pd.read_csv('X_train_transformed.csv')
X_test = pd.read_csv('X_test_transformed.csv')
y_train_clf = pd.read_csv('y_train_clf.csv').squeeze()
y_test_clf = pd.read_csv('y_test_clf.csv').squeeze()
y_train_reg = pd.read_csv('y_train_reg.csv').squeeze()
y_test_reg = pd.read_csv('y_test_reg.csv').squeeze()

# للتصنيف، يُفضل إزالة عمود Overall_Rating (كما فعلت في كود Logistic Regression)
X_train_clf = X_train.drop('num__Overall_Rating', axis=1)
X_test_clf = X_test.drop('num__Overall_Rating', axis=1)

In [ ]:
from sklearn.linear_model import LogisticRegression

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'newton-cg', 'saga']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid_lr,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train_clf, y_train_clf)

print("Best Logistic Regression Params:", lr_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(lr_grid.best_score_))
best_lr = lr_grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_clf)
print("Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_lr)))

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Logistic Regression Params: {'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Accuracy: 0.8617
Test Accuracy: 0.8627


In [ ]:
from sklearn.tree import DecisionTreeClassifier

param_grid_dt_clf = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_clf_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid_dt_clf,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

dt_clf_grid.fit(X_train_clf, y_train_clf)

print("Best Decision Tree Classifier Params:", dt_clf_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(dt_clf_grid.best_score_))
best_dt_clf = dt_clf_grid.best_estimator_
y_pred_dt_clf = best_dt_clf.predict(X_test_clf)
print("Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_dt_clf)))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best Decision Tree Classifier Params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV Accuracy: 0.8873
Test Accuracy: 0.8854


In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf_clf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf_clf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid_rf_clf,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

rf_clf_grid.fit(X_train_clf, y_train_clf)

print("Best Random Forest Classifier Params:", rf_clf_grid.best_params_)
print("Best CV Accuracy: {:.4f}".format(rf_clf_grid.best_score_))
best_rf_clf = rf_clf_grid.best_estimator_
y_pred_rf_clf = best_rf_clf.predict(X_test_clf)
print("Test Accuracy: {:.4f}".format(accuracy_score(y_test_clf, y_pred_rf_clf)))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Random Forest Classifier Params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100}
Best CV Accuracy: 0.9045
Test Accuracy: 0.9034


In [ ]:
from sklearn.tree import DecisionTreeRegressor

param_grid_dt_reg = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_reg_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_dt_reg,
    cv=KFold(5),
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

dt_reg_grid.fit(X_train, y_train_reg)

print("Best Decision Tree Regressor Params:", dt_reg_grid.best_params_)
print("Best CV R²: {:.4f}".format(dt_reg_grid.best_score_))
best_dt_reg = dt_reg_grid.best_estimator_
y_pred_dt_reg = best_dt_reg.predict(X_test)
print("Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_dt_reg)))
print("Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_dt_reg))))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Decision Tree Regressor Params: {'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 5}
Best CV R²: 0.9522
Test R²: 0.9382
Test RMSE: 1.5765


In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf_reg = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf_reg_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid_rf_reg,
    cv=KFold(5),
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

rf_reg_grid.fit(X_train, y_train_reg)

print("Best Random Forest Regressor Params:", rf_reg_grid.best_params_)
print("Best CV R²: {:.4f}".format(rf_reg_grid.best_score_))
best_rf_reg = rf_reg_grid.best_estimator_
y_pred_rf_reg = best_rf_reg.predict(X_test)
print("Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_rf_reg)))
print("Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Random Forest Regressor Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
Best CV R²: 0.9684
Test R²: 0.9672
Test RMSE: 1.1484


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline

# تحضير البيانات باستخدام أفضل درجة من الكود الأصلي (يمكنك تغييرها)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# === Ridge ===
param_grid_ridge = {'alpha': np.logspace(-3, 3, 7)}
ridge_grid = GridSearchCV(Ridge(random_state=42), param_grid_ridge, cv=5, scoring='r2', n_jobs=-1, verbose=1)
ridge_grid.fit(X_train_poly, y_train_reg)
print("Best Ridge alpha:", ridge_grid.best_params_['alpha'])
print("Best CV R²: {:.4f}".format(ridge_grid.best_score_))
best_ridge = ridge_grid.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_poly)
print("Ridge Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_ridge)))

# === Lasso ===
lasso_grid = GridSearchCV(Lasso(random_state=42, max_iter=5000), param_grid_ridge, cv=5, scoring='r2', n_jobs=-1, verbose=1)
lasso_grid.fit(X_train_poly, y_train_reg)
print("Best Lasso alpha:", lasso_grid.best_params_['alpha'])
print("Best CV R²: {:.4f}".format(lasso_grid.best_score_))
best_lasso = lasso_grid.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_poly)
print("Lasso Test R²: {:.4f}".format(r2_score(y_test_reg, y_pred_lasso)))

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Ridge alpha: 100.0
Best CV R²: 0.7554
Ridge Test R²: 0.7655
Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Lasso alpha: 0.001
Best CV R²: 0.7552
Lasso Test R²: 0.7590


In [ ]:
results = []
results.append(['Logistic Regression', lr_grid.best_score_, accuracy_score(y_test_clf, y_pred_lr)])
results.append(['Decision Tree Classifier', dt_clf_grid.best_score_, accuracy_score(y_test_clf, y_pred_dt_clf)])
results.append(['Random Forest Classifier', rf_clf_grid.best_score_, accuracy_score(y_test_clf, y_pred_rf_clf)])
results.append(['Decision Tree Regressor', dt_reg_grid.best_score_, r2_score(y_test_reg, y_pred_dt_reg)])
results.append(['Random Forest Regressor', rf_reg_grid.best_score_, r2_score(y_test_reg, y_pred_rf_reg)])
results.append(['Ridge Regression (poly)', ridge_grid.best_score_, r2_score(y_test_reg, y_pred_ridge)])
results.append(['Lasso Regression (poly)', lasso_grid.best_score_, r2_score(y_test_reg, y_pred_lasso)])

df_results = pd.DataFrame(results, columns=['Model', 'Best CV Score', 'Test Score'])
print(df_results)

                      Model  Best CV Score  Test Score
0       Logistic Regression       0.861692    0.862735
1  Decision Tree Classifier       0.887307    0.885358
2  Random Forest Classifier       0.904532    0.903406
3   Decision Tree Regressor       0.952240    0.938208
4   Random Forest Regressor       0.968424    0.967213
5   Ridge Regression (poly)       0.755356    0.765488
6   Lasso Regression (poly)       0.755216    0.758962
